## **Setup**

In [ ]:
# Notebook autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
# Path setup
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
# Libraries
import os as os
import json as json
import torch
import matplotlib.pyplot as plt

# Local
from chatGnT.config import CFG, ensure_dirs
from chatGnT.data import load, preprocess, tokenize, dataloaders
from chatGnT.models import train

# Setup
ensure_dirs(CFG)

## **Load Tokens & Vocabs**

In [ ]:
with open(CFG.outputs_dir / "models/tokens_mt.json", "r") as f:
    tokens_mt = json.load(f)
with open(CFG.outputs_dir / "models/vocab_mt_amt.json", "r") as f:
    vocab_amt = json.load(f)
with open(CFG.outputs_dir / "models/vocab_mt_ingred.json", "r") as f:
    vocab_ingred = json.load(f)

inv_vocab_amt, inv_vocab_ingred = tokenize.invert_vocab_mt(vocab_amt, vocab_ingred)
tokens_padded = tokenize.encode_tokens_mt(tokens_mt, vocab_amt, vocab_ingred)

In [ ]:
print("Amount vocab ntokens:", len(vocab_amt))
print("First 6:", list(vocab_amt.items())[:6])
print("Last 3:", list(vocab_amt.items())[-3:])

print("Ingredient vocab ntokens:", len(vocab_ingred))
print("First 6:", list(vocab_ingred.items())[:6])
print("Last 3:", list(vocab_ingred.items())[-3:])

In [ ]:
# Unzip the padded tokens
amt_seqs, ingred_seqs = zip(*tokens_padded)

# Convert to tensors
amt_tensor = torch.tensor(amt_seqs, dtype=torch.long)
ingred_tensor = torch.tensor(ingred_seqs, dtype=torch.long)

# Check shape: (num_recipes, seq_len)
print(amt_tensor.shape, ingred_tensor.shape)

## **Model Config**

In [ ]:
config = {
    # model architecture
    "ntoken_amt": len(vocab_amt),
    "ntoken_ingred": len(vocab_ingred),
    "ninp": 64,  # embed size
    "nhead": 2,
    "nhid": 64,  # feed forward size
    "nlayers": 2,
    "model_version": "multi_task",
    "batch_size": 16,

    # training setup
    "learning_rate": 1e-3,
    "weight_decay": 1e-3,

    # scheduler
    "scheduler_type": "reduce_on_plateau",  # "step" or "reduce_on_plateau"
    # reduce on plateau scheduler settings
    "scheduler_factor": 0.5,
    "scheduler_patience": 4,

    # token details
    "pad_id_amt": vocab_amt["<pad>"],
    "pad_id_ingred": vocab_ingred["<pad>"],

    # dataloaders
    "split": 0.85,
    "seed": 42,

    # epochs, early stopping, & logs
    "epochs": 500,
    "early_stop": 50,
    "log_interval": 6
}

## **Train Model**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

result = train.run_training(
    config,
    {
        "amt_tensor": amt_tensor,
        "ingred_tensor": ingred_tensor,
    },
    device,
)

best_model = result["best_model"]
train_losses = result["train_losses"]
val_losses = result["val_losses"]
gradient_magnitudes = result["gradient_magnitudes"]


In [ ]:
train.plot_training_history(train_losses, val_losses, gradient_magnitudes)


In [ ]:
train.save_multi_task_artifacts(best_model, config, vocab_amt, vocab_ingred, train_losses, val_losses)
